In [1]:
# install dependencies

!pip install -q -U uv
!uv pip install --system "ai-edge-torch==0.7.1" "tensorflow==2.20.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 72.8 MB/s eta 0:00:00:00:0100:01
Using Python 3.12.12 environment at: /usr
Resolved 87 packages in 561ms                                        
Prepared 12 packages in 12.01s                                           
Uninstalled 4 packages in 2.33s
Installed 12 packages in 10.01s                             
 + ai-edge-litert==2.1.2
 + ai-edge-quantizer==0.4.2
 + ai-edge-tensorflow==2.21.0.dev20251110
 + ai-edge-torch==0.7.1
 + backports-strenum==1.3.1
 - h5py==3.15.1
 + h5py==3.14.0
 + keras-nightly==3.14.0.dev2026032304
 - protobuf==5.29.5
 + protobuf==7.34.1
 + tb-nightly==2.20.0a20250717
 - tensorboard==2.19.0
 + tensorboard==2.20.0
 - tensorflow==2.19.0
 + tensorflow==2.20.0
 + torch-xla2==0.0.1.dev202412041639


In [2]:
# Disable gpu variables

import os

os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["JAX_PLATFORMS"] = "cpu"

In [3]:
import torch
import ai_edge_torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

/usr/local/lib/python3.12/dist-packages/torch/distributed/distributed_c10d.py:351: UserWarning: Device capability of jax unspecified, assuming `cpu` and `cuda`. Please specify it via the `devices` argument of `register_backend`.
  warnings.warn(
ERROR:2026-03-23 09:47:30,214:jax._src.xla_bridge:487: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jax/_src/xla_bridge.py", line 485, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/usr/local/lib/python3.12/dist-packages/jax_plugins/xla_cuda12/__init__.py", line 328, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/usr/local/lib/python3.12/dist-packages/jax_plugins/xla_cuda12/__init__.py", line 285, in _check_cuda_versions
    local_device_count = cuda_versions.cuda_device_count()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: jaxlib/cuda/versions_h

In [4]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

In [5]:
def setup_authentication() -> bool:
    try:
        from kaggle_secrets import UserSecretsClient
        from huggingface_hub import login

        user_secrets = UserSecretsClient()
        hf_token = "your_api_key"
        if hf_token:
            login(token=hf_token)
        return bool(hf_token)
    except Exception as e:
        print(f"Authentication failed. Must for Pvt Repo {e}")
        return False


setup_authentication()

True

In [6]:
REPO = "apps1/medical_v7"
OUTPUT_NAME = "overall_mini_medical_V7"
SAMPLE_TEXT = "Who are you?"

In [7]:
# Load model
orig_model = AutoModelForSequenceClassification.from_pretrained(
    REPO, trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(REPO, trust_remote_code=True)

config.json:   0%|          | 0.00/788 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/19.1M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

In [8]:
classification_pipeline = pipeline(
    "text-classification", model=orig_model, tokenizer=tokenizer
)


class MyModelWrapper(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = classification_pipeline.model

    def forward(self, input_ids, attention_mask=None):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.logits

Device set to use cpu


In [9]:
def get_ids_and_mask(text: str):
    tokens = tokenizer(
        text, return_tensors="pt", padding="max_length", max_length=512, truncation=True
    )
    ids_and_mask = (tokens["input_ids"], tokens["attention_mask"])
    return ids_and_mask

In [10]:
imput = get_ids_and_mask(SAMPLE_TEXT)

edge_model = ai_edge_torch.convert(MyModelWrapper().eval(), imput)
edge_model.export(f"{OUTPUT_NAME}.tflite")

2026-03-23 09:48:44.605884: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


INFO:tensorflow:Assets written to: /tmp/tmp53yxq4tw/assets


INFO:tensorflow:Assets written to: /tmp/tmp53yxq4tw/assets
W0000 00:00:1774259326.660790      55 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1774259326.660833      55 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1774259326.677412      55 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


# Infer

In [12]:
tflite_model = ai_edge_torch.load(f"{OUTPUT_NAME}.tflite")

In [13]:
test_input = "Grab 00 crispy pcs for Rs.000 with KFC Wednesday Specials! 0 Strips, 0 Wings and 0 Hot n Crispy inside. Order now! bit.ly/kfcind-wdga00 TnC"

In [14]:
def print_info(logits) -> None:
    probabilities = torch.nn.functional.softmax(logits, dim=1)

    print("Output logits shape:", logits.shape)
    print("Output logits:", logits)
    print("Output probabilities:", probabilities)

In [15]:
input = get_ids_and_mask(test_input)

In [16]:
import pandas as pd
import torch
import numpy as np
from sklearn.metrics import classification_report, accuracy_score

# Load data
csv = pd.read_csv("/kaggle/input/datasets/aiat2025/augmentedmedicaldata/augmented_dataset12.csv").tail(1000)

texts = csv["augmented_text"].dropna()
labels = csv.loc[texts.index, "label"]

texts = texts.tolist()
labels = labels.tolist()

print("Total samples:", len(texts))

BATCH_SIZE = 32

y_true = []
y_pred = []

for i in range(0, len(texts), BATCH_SIZE):
    batch_texts = texts[i:i+BATCH_SIZE]
    batch_labels = labels[i:i+BATCH_SIZE]

    for text, true_label in zip(batch_texts, batch_labels):
        ids, mask = get_ids_and_mask(text)

        # ensure correct shape (VERY IMPORTANT)
        ids = ids.unsqueeze(0) if ids.dim() == 1 else ids
        mask = mask.unsqueeze(0) if mask.dim() == 1 else mask

        # TFLite inference (single sample only)
        tflite_logits = tflite_model(ids, mask)

        logits = torch.tensor(tflite_logits)
        pred_class = torch.argmax(logits, dim=-1).item()

        y_pred.append(pred_class)
        y_true.append(true_label)
        if pred_class != true_label:
            print(text)
            print(pred_class,true_label)

    # optional cleanup
    torch.cuda.empty_cache()

# Metrics
y_true = np.array(y_true)
y_pred = np.array(y_pred)

accuracy = accuracy_score(y_true, y_pred)
print(f"\nAccuracy: {accuracy:.4f}")

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

Total samples: 998
Tab. Ibuprofeno 400 mg dos dia
Paciente: John Smith
Idade: 18
Diagnóstico: Febre viral
Instrucciones: Beba líquidos y descansar
Laboratório: City Hospital
2 0


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
ERROR: Subgraph reshaping is enabled by default, TFLITE_XNNPACK_DELEGATE_FLAG_ENABLE_SUBGRAPH_RESHAPING is deprecated and will be removed in the future.


Migraine
Patient: John Smith (Age: 20, Gender: M)
Lab: MediCare Diagnostics
Instructions: Drink fluids and rest
Prescribed: 1 Tab Pantoprazole 40 m g once da ily (typo: 'm g' and 'da ily')
2 0
BioSphere Advanced Pathology LABORATORY TEST REPORT
Collected: 2024-12-16
Glucose Random(196.63cells/cumm)
Serum Creatinine(322.21cells/cumm)
Lab Internal QC OK
PatientID: 368599
Total WBC:308.07fL
Potassium-187.65 fL
Hemoglobin-370.55 %
Blood Urea:397.53%
Signature: Dr. R. Kumar
1 0
Dr. S. Patel, MediLab Pathology Services: Tab Glocose 100 mg OD, Patient Meena Singh, Age: 30 years, Sex: F, City: Mumbai. Advise: Drink fluids, rest.
2 0
Metro Care Center
Meena Singh
Age: 41 yrs
Gender: F
Medicine: Tab, Cefixime 200mg OD
Dosage: 100mg BID after meals
Advice: Drink plenty of fluids and rest
2 0
Metro Care Center →
Lab Report: Blood Glucose (Glocose) Random: 196.63 cells/cumm
WBC: 308.07 fL (Total)
Serum Creatinine: 322.21 cells/cumm
Collected: 2024-12-16
Patient: Meena Singh, Age: 51 years
Doctor: D

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [17]:
# original
with torch.no_grad():
    original_logits = orig_model(*input).logits

    print("\nOriginal Inference successful!")
    print_info(original_logits)


Original Inference successful!
Output logits shape: torch.Size([1, 3])
Output logits: tensor([[-1.1262, -1.4582,  3.0692]])
Output probabilities: tensor([[0.0147, 0.0105, 0.9748]])


In [18]:
diff = torch.abs(torch.tensor(tflite_logits) - original_logits).max().item()
print(f"\nMax difference between TFLite and PyTorch logits: {diff:.8f}")


Max difference between TFLite and PyTorch logits: 4.13138723
